# Tarea 3 — Elementos Finitos en Diseño Mecánico

## Problema 1


In [1]:
import numpy as np


In [2]:
def funciones_forma(r,s):
    N1 = 1 - s - r
    N2 = r
    N3 = s

    dN = np.array([[-1, -1], [1, 0], [0, 1]])

    return N1, N2, N3, dN


def matriz_constitutiva(E, nu):
    D = np.array([[E*(1-nu)/((1+nu)*(1-2*nu)), E*nu/((1+nu)*(1-2*nu)), E*nu/((1+nu)*(1-2*nu)), 0],
                  [E*nu/((1+nu)*(1-2*nu)), E*(1-nu)/((1+nu)*(1-2*nu)), E*nu/((1+nu)*(1-2*nu)), 0],
                  [E*nu/((1+nu)*(1-2*nu)), E*nu/((1+nu)*(1-2*nu)), E*(1-nu)/((1+nu)*(1-2*nu)), 0],
                  [0, 0, 0, E/(2*(1+nu))]])

    return D


def matriz_jacobiana(coord, dN):
    coord = np.array(coord)
    J = coord.T @ dN
    detJ = np.linalg.det(J)
    invJ = np.linalg.inv(J)

    return J, detJ, invJ


def matriz_B(N, dN, invJ, radio):
    B = np.zeros((4, 6))
    for i in range(3):
        B[0, 2*i] = dN[i, 0] * invJ[0, 0] + dN[i, 1] * invJ[1, 0]
        B[1, 2*i+1] = dN[i, 0] * invJ[0, 1] + dN[i, 1] * invJ[1, 1]
        B[2, 2*i] = N[i]/radio
        B[3, 2*i] = dN[i, 0] * invJ[0, 1] + dN[i, 1] * invJ[1, 1]
        B[3, 2*i+1] = dN[i, 0] * invJ[0, 0] + dN[i, 1] * invJ[1, 0]
    return B


def rigidez_elemental(coord, D):
    coord = np.array(coord)
    puntos = [[1/6, 1/6], [2/3, 1/6], [1/6, 2/3]]
    pesos = [1/6, 1/6, 1/6]
    Ke = np.zeros((6, 6))

    for k in range(3):
        r, s = puntos[k]
        N1, N2, N3, dN = funciones_forma(r, s)
        N = np.array([N1, N2, N3])
        radio = N1*coord[0, 0] + N2*coord[1, 0] + N3*coord[2, 0]
        J, detJ, invJ = matriz_jacobiana(coord, dN)
        B = matriz_B(N, dN, invJ, radio)
        Ke += 2*np.pi * pesos[k] * B.T @ D @ B * radio * abs(detJ)

    return Ke


def gdl_elemento(nodos):
    gdl = []
    for i in nodos:
        gdl.append(2*i)
        gdl.append(2*i+1)

    return gdl


def ensamblar_rigidez_global(coordenadas, conectividad, E, nu):
    coordenadas = np.array(coordenadas)
    n_nodos = len(coordenadas)
    n_gdl = 2*n_nodos
    K = np.zeros((n_gdl, n_gdl))
    D = matriz_constitutiva(E, nu)

    for nodos in conectividad:
        coord = coordenadas[nodos]
        Ke = rigidez_elemental(coord, D)
        gdl = gdl_elemento(nodos)
        K[np.ix_(gdl, gdl)] += Ke

    return K


def resolver_sistema(K, F, gdl_restringidos):
    K = np.array(K)
    F = np.array(F)
    n_gdl = len(F)
    gdl_libres = []

    for i in range(n_gdl):
        if i not in gdl_restringidos:
            gdl_libres.append(i)

    K_libre = K[np.ix_(gdl_libres, gdl_libres)]
    F_libre = F[gdl_libres]
    U_libres = np.linalg.solve(K_libre, F_libre)

    U = np.zeros(n_gdl)
    U[gdl_libres] = U_libres
    R = K @ U - F

    return U, R


def calcular_tensiones(coords_elem, U_elem, D, puntos):
    coords_elem = np.array(coords_elem)
    sigma = np.zeros((len(puntos), 4))

    for k in range(len(puntos)):
        r, s = puntos[k]
        N1, N2, N3, dN = funciones_forma(r, s)
        N = np.array([N1, N2, N3])
        radio = N1*coords_elem[0, 0] + N2*coords_elem[1, 0] + N3*coords_elem[2, 0]
        J, detJ, invJ = matriz_jacobiana(coords_elem, dN)
        B = matriz_B(N, dN, invJ, radio)
        epsilon = B @ U_elem
        sigma[k] = D @ epsilon

    return sigma


def von_mises(sigma):
    sigma_rr = sigma[0]
    sigma_zz = sigma[1]
    sigma_tt = sigma[2]
    tau_rz = sigma[3]

    sigma_vm = np.sqrt(((sigma_rr-sigma_zz)**2 + (sigma_zz-sigma_tt)**2 + (sigma_tt-sigma_rr)**2)/2 + 3*tau_rz**2)

    return sigma_vm


## Prueba: cilindro uniforme sometido a tracción

Se usa un cilindro de radio 10 mm y longitud 100 mm, con E = 210000 MPa y nu = 0.3. La tracción axial uniforme es de 100 MPa. La sección meridional se divide en dos triángulos.

En el eje se impone ur = 0 y en la base uz = 0, dejando libre la contracción radial de la base. La solución analítica es:

- ur = -nu * sigma_0 * r / E
- uz = sigma_0 * z / E
- [sigma_rr, sigma_zz, sigma_tt, tau_rz] = [0, sigma_0, 0, 0]
- Von Mises = sigma_0 y Kt = 1.

Las fuerzas equivalentes de la cara superior se calculan integrando la tracción con el factor 2*pi*r: P/3 en el nodo del eje y 2*P/3 en el nodo exterior.


In [3]:
E = 210000  # MPa
nu = 0.3
radio = 10  # mm
L = 100  # mm
sigma_0 = 100  # MPa
P = sigma_0*np.pi*radio**2  # N

# Coordenadas [r, z] e indices de nodos desde cero.
coordenadas = np.array([[0, 0],
                        [radio, 0],
                        [0, L],
                        [radio, L]])

conectividad = np.array([[0, 1, 3],
                         [0, 3, 2]])

F = np.zeros(2*len(coordenadas))
F[5] = P/3
F[7] = 2*P/3

# ur = 0 en nodos 0 y 2; uz = 0 en nodos 0 y 1.
gdl_restringidos = [0, 4, 1, 3]


In [4]:
K = ensamblar_rigidez_global(coordenadas, conectividad, E, nu)
U, R = resolver_sistema(K, F, gdl_restringidos)
D = matriz_constitutiva(E, nu)

puntos = [[1/6, 1/6], [2/3, 1/6], [1/6, 2/3]]
tensiones = np.zeros((len(conectividad), len(puntos), 4))
sigma_vm = np.zeros((len(conectividad), len(puntos)))

for e in range(len(conectividad)):
    nodos = conectividad[e]
    coords_elem = coordenadas[nodos]
    gdl = gdl_elemento(nodos)
    U_elem = U[gdl]
    tensiones[e] = calcular_tensiones(coords_elem, U_elem, D, puntos)

    for k in range(len(puntos)):
        sigma_vm[e, k] = von_mises(tensiones[e, k])


In [5]:
U_analitico = np.zeros(len(U))
for i in range(len(coordenadas)):
    r, z = coordenadas[i]
    U_analitico[2*i] = -nu*sigma_0*r/E
    U_analitico[2*i+1] = sigma_0*z/E

sigma_analitica = np.array([0, sigma_0, 0, 0])
error_U = np.max(np.abs(U - U_analitico))
error_sigma = np.max(np.abs(tensiones - sigma_analitica))
error_vm = np.max(np.abs(sigma_vm - sigma_0))
error_simetria = np.max(np.abs(K - K.T))/np.max(np.abs(K))
reaccion_axial = R[1] + R[3]
error_equilibrio = abs(reaccion_axial + P)/P
Kt = np.max(sigma_vm)/sigma_0

print("Nodo      ur MEF       ur analitico       uz MEF       uz analitico [mm]")
for i in range(len(coordenadas)):
    print(f"{i:4d} {U[2*i]:13.6e} {U_analitico[2*i]:17.6e} {U[2*i+1]:13.6e} {U_analitico[2*i+1]:17.6e}")

print("\nTensiones [rr, zz, tt, rz] en el primer punto de cada elemento [MPa]:")
for e in range(len(conectividad)):
    print(f"Elemento {e}: {tensiones[e, 0]}")

print(f"\nError maximo de desplazamientos: {error_U:.3e} mm")
print(f"Error maximo de tensiones: {error_sigma:.3e} MPa")
print(f"Error maximo de Von Mises: {error_vm:.3e} MPa")
print(f"Error relativo de simetria de K: {error_simetria:.3e}")
print(f"Carga aplicada: {P:.6f} N")
print(f"Reaccion axial: {reaccion_axial:.6f} N")
print(f"Error relativo de equilibrio axial: {error_equilibrio:.3e}")
print(f"Kt del cilindro uniforme: {Kt:.6f}")

assert np.allclose(U, U_analitico, rtol=1e-8, atol=1e-10)
assert np.allclose(tensiones, sigma_analitica, rtol=1e-8, atol=1e-8)
assert np.allclose(sigma_vm, sigma_0, rtol=1e-8, atol=1e-8)
assert error_simetria < 1e-10
assert error_equilibrio < 1e-10
print("Prueba del cilindro uniforme: correcta.")


Nodo      ur MEF       ur analitico       uz MEF       uz analitico [mm]
   0  0.000000e+00     -0.000000e+00  0.000000e+00      0.000000e+00
   1 -1.428571e-03     -1.428571e-03  0.000000e+00      0.000000e+00
   2  0.000000e+00     -0.000000e+00  4.761905e-02      4.761905e-02
   3 -1.428571e-03     -1.428571e-03  4.761905e-02      4.761905e-02

Tensiones [rr, zz, tt, rz] en el primer punto de cada elemento [MPa]:
Elemento 0: [0.00000000e+00 1.00000000e+02 0.00000000e+00 1.36828399e-16]
Elemento 1: [  0. 100.   0.   0.]

Error maximo de desplazamientos: 4.163e-17 mm
Error maximo de tensiones: 8.527e-14 MPa
Error maximo de Von Mises: 8.527e-14 MPa
Error relativo de simetria de K: 2.321e-17
Carga aplicada: 31415.926536 N
Reaccion axial: -31415.926536 N
Error relativo de equilibrio axial: 1.158e-15
Kt del cilindro uniforme: 1.000000
Prueba del cilindro uniforme: correcta.
